# 🦾🧠 **MLOps on RHOAI: 模型服务 & KServe & Knative**

## 📑 **0. 目录**

- ▶️ 1. 模型服务 & 模型服务运行时
- 💿 2. 模型的存储格式 
- 💾 3. 使用模型服务器进行模型部署
- 🌐 4. 单模型服务与多模型服务中模型推理 API 的生成
- 🔎 5. 参考链接

## ▶️ **1. 模型服务 & 模型服务运行时**

- 机器学习（ML）**模型服务**（model serving）是指部署、管理与公开 ML 模型以进行推理的过程，这使得模型能够用作针对未见数据的推理服务。
- 模型服务的关键概念包括：
  - 部署（deplyment）
  - 扩展（scalability）
  - 实时推理（real-time inference）：在一些场景中是常见的，如诈骗检测、图像识别与自然语言处理。
  - 批量推理（batch inference）：推断请求在真实系统中延迟可能提高到一定程度，这在生产环境中是不合适的。以批量化地构建许多请求降低通信开销（成本）。在一些场景中是常见的，如数据预处理、推荐系统、分析系统和大规模数据转换。
  - 监控与日志（monitoring and logging）
  - 版本控制（versioning）
  - 安全（security）
  - 整合（integration）
- **模型服务运行时**（model serving runtime）或 **模型服务器**（model server）是运行训练好的机器学习模型来推理预测的执行环境。一个模型服务器定义标准的 HTTP 或 gRPC 端点，加载模型至内存中，处理客户端请求，执行推理，与返回结果。
- 模型服务运行时可作为大型部署平台的一部分，如 **KServe**，它包含的特性诸如可扩展性、版本化、监控与安全。模型服务运行时包括 TensorFlow Serving, TorchServe, **OpenVINO** 与 **ONNX 运行时（runtime）**。这些运行时支持由流行机器学习框架训练的模型格式的部署。有些模型只能与来自于单一机器学习框架的模型工作，而其他模型服务器支持多个框架。

## 💿 **2. 模型的存储格式**

- 将模型发布提供推理服务或在团队内以及公开共享，实现这些场景的第一步是将其序列化并保存至磁盘上。
- 每个机器学习框架提供它自身的格式用于保存和加载模型。模型的导出方法与格式应该与用来运行模型的运行时保持兼容性。

### **2.1 保存 Scikit-learn 模型**

In [3]:
import joblib  # 导入 joblib 模块

保存模型：逻辑回归模型定义中 `joblib.dump()` 函数的第一个实参可以是 `model` 也可以是 `pipe`。

加载 joblib 序列化的模型文件

🎇 注意：使用 **jiblib** 进行序列化保存的模型无法在 RHOAI 中部署运行。为了运行此种格式，可以构建自定义的模型服务器 **MLServer** 来支持运行。

### **2.2 保存 TensorFlow 模型**

TensorFlow 能以不同的格式保存模型：

- SavedModel：此格式以目录方式保存模型，其中包含了模型的 **Protobuf** 二进制，以 `.pb` 作为扩展名。OpenVINO 模型服务器支持此种格式。
- Keras v3：以 `.keras` 作为扩展名，使用 Keras API 的话更推荐此种格式。OpenVINO 模型服务器不支持此种格式。
- HDF5：早期的 TensorFlow 1 兼容的格式，以 `.h5` 作为扩展名。

TensorFlow Keras API 使用模型对象实例（在程序上下文中已定义）的 `save` 方法以及传递输出的路径作为参数进行模型保存。不同的路径代表不同的格式，如下所示：

加载模型的 3 种方式，如下所示：

### **2.3 保存 PyTorch 模型**

### **2.4 ONNX**

#### **2.4.1 从 Scikit-learn 到 ONNX**

#### **2.4.2 从 TensorFlow 到 ONNX**

#### **2.4.3 从 PyTorch 到 ONNX**

## 💾 **3. 使用模型服务器进行模型部署**

📢 说明：下文中 serving runtime (服务运行时)、mode serving runtime (模型服务运行时)、model server (模型服务器)、inference server (推理服务器)

- RHOAI 使用 KServe 作为模型服务（model serving）与推理平台（inferencing platform）。**KServe** 是一个 Kubernetes 平台，使机器学习模型在云环境中部署更加简便。
- RHOAI 随 **Red Hat OpenShift Serverless** 与 **Red Hat OpenShift Service Mesh** 提供给 KServe，用来处理扩展与通信。
- 🔀 RHOAI 提供模型服务的平台包括两类：
  - 单模型服务（single-model serving）
  - 多模型服务（multi-model serving）
- RHOAI 模型服务的自定义资源定义：
  - **serving.kserve.io.ServingRuntime CRD**：模型服务器作为其实例。ServingRuntime 对象定义了模型服务器 Pods 的模板，它加载模型至内存中、处理客户端请求、执行推断并返回结果。
  - **serving.kserve.io.InferenceService CRD**：InferenceService 对象定义 REST 或 gRPC APIs 来公开模型。

### **3.1 单模型服务（single-model serving）**

#### **3.1.1 架构说明**

- 在模型服务器中仅仅支持服务单个模型
- ServingRuntime 对象使用以下的容器来配置模型服务器 Pods 以服务模型：
  - **storage-initializer** (kserve-storage-initializer)：初始化容器，为 kserve-container 容器下载拉取模型。
  - **kserve-container** (openvino_model_server)：可运行服务运行时（serving runtime）的容器镜像（可运行各类模型服务器）。
  - **<span style="color:red">queue-proxy**</span> (serving-queue)：可运行 *serverless* 特性的容器镜像
  - **istio-proxy** (proxyv2)：在 Red Hat OpenShift Service Mesh control 下运行 HTTP 服务的容器镜像

<center><img src="images/rhoai-single-model-pod.png" style="width:80%"></center>

<center>图例：项目中单模型服务 Pod 中的容器镜像</center>

- 多个服务运行时已集成至 RHOAI Dashboard 中（Pre-installed）

<center><img src="images/RHOAI-Serving-runtimes-pre-installed.png" style="width:80%"></center>

<center>图例：oc get template -n redhat-ods-applications (获取指定项目中所有的服务运行时模板)</center>

- OpenVINO 模型服务器：
  - 支持的模型格式：**`OpenVINO IR`**、**`ONNX`**、TensorFlow 1/2、Paddle、PyTorch
  - OpenVINO 模型服务器对 Scikit-learn 模型不是完全兼容的，对于此类模型更推荐使用 **Nvidia Triton 推理服务器** 或自定义 **MLserver**。
- 面向 KServe 的 Caikit 独立服务运行时
- TGIS 独立服务运行时
- 面向 KServe 的 Caikit TGIS 服务运行时
- 面向 KServe 的 vLLM 服务运行时

#### **3.1.2 单模型服务部署方法**

<center><img src="images/rhoai-single-model-deploy-1.png" style="width:80%"></center>

<center><img src="images/model-serving-secret.png" style="width:80%"</center>

<center><img src="images/rhoai-single-model-deploy-2.png" style="width:80%"></center>

<center><img src="images/rhoai-single-model-deploy-3.png" style="width:80%"></center>

<center>图例：单模型服务部署成功（Status 为 Loaded 状态）</center>

#### **3.1.3 KServe 与 Knative 在单模型服务中的协作时序**

<center><img src="images/kserve-knative-rhoai-singlemodel-cooperate.png" style="width:100%"></center>

<center>图例：KServe + Knative 在 OpenShift AI 单模型服务中的协作时序</center>

| 关键步骤 | 机制原理 |
| ----- | ----- |
| 1️⃣ 控制面 | KServe Controller 生成 `Knative Service CR`（knative-serving CR），queue-proxy 必注入。|
| 2️⃣ 缩 0 | queue-proxy 每 100 ms 上报 0 并发 → `Autoscaler` 置副本 0 |
| 3️⃣ 冷启动 | `Activator` 缓存请求并触发扩容；queue-proxy 就绪后反向通知 → 瞬间转发 |
| 4️⃣ 运行 | queue-proxy 8012 端口每 100 ms 上报并发，Autoscaler 线性扩容。|
| 5️⃣ 再缩 0 | 无流量时 queue-proxy 持续 0 → Autoscaler 置 0 |


### **3.2 多模型服务（multi-model serving）**

#### **3.2.1 架构说明**

<center><img src="images/containers-single-and-multi-model-serving.png" style="width:100%"></center>

<center>图例：RHOAI 中单模型服务 vs. 多模型服务</center>

在多模型平台中，服务运行时对 pod 进行配置，以使⽤以下容器为模型提供服务：

- **rest-proxy** (odh-mm-rest-proxy)：运行 HTTP 代理，将推理请求重定向到 gRPC 端点，即 HTTP ➡ gRPC。 
- **oauth-proxy** (ose-oauth-proxy)：运行 OAuth 代理，将模型访问权限与 OpenShift ⾝份验证系统集成。 
- **ovms** (openvino_model_server)：运行 OpenVINO 模型服务器。 
- **ovms-adapter** (odh-modelmesh-runtime-adapter)：KServe 和 OpenVINO 模型服务器之间的媒介，从 S3 存储桶拉取加载模型。 
- **mm** (odh-modelmesh)：model mesh 编排模型放置并路由 gRPC 推理请求的模型网格容器，即 gRPC ➡ 模型运行时。

<center><img src="images/rhoai-multi-model-pod.png" style="width:80%"></center>

<center>图例：项目中多模型服务 Pod 中的容器镜像</center>

#### **3.2.2 多模型服务中的 ModelMesh**

ModelMesh 架构默认关闭了 Knative Serving 的 `queue-proxy` 注入，转而使用 **纯 Kubernetes Deployment + 共享网络命名空间** 的方式运行模型 Pod，其中包含 modelmesh-runtime + puller 等容器（如上图所示），因此看不到 queue-proxy，并不等于不具备 Serverless 能力，而是被简化/换掉。

单模型服务（KServe）才默认启用 Knative Serving（queue-proxy 存在）。多模型服务（ModelMesh）为了 **高密度共享 Pod** 和 **减少 Sidecar 开销**，默认把 Knative 注入关掉。

#### **3.2.3 多模型服务部署方法**

多模型服务器部署之前需创建模型服务器，再在其中部署多个模型。如下所示：

<center><img src="images/rhoai-multi-model-deploy-1.png" style="width:80%"></center>

<center><img src="images/rhoai-multi-model-deploy-2.png" style="width:80%"></center>

<center><img src="images/rhoai-multi-model-deploy-3.png" style="width:80%"></center>

<center>图例：添加模型服务器</center>

<center><img src="images/rhoai-multi-model-deploy-4.png" style="width:80%"></center>

<center>图例：模型服务器添加成功，待部署模型</center>

<center><img src="images/rhoai-multi-model-deploy-5.png" style="width:80%"></center>

<center><img src="images/minio-bucket-models.png" style="width:80%"></center>

<center>图例：指定参数部署模型</center>

<center><img src="images/rhoai-multi-model-deploy-6.png" style="width:80%"></center>

<center><img src="images/rhoai-multi-model-deploy-7.png" style="width:80%"></center>

<center>图例：模型在多模型服务中部署成功（模型推理端点与连接 token）</center>

- 如下所示，在指定项目中部署多模型服务后创建的 ServingRuntimes 与 InferenceServices 自定义资源：

<center><img src="images/model-serving-cr-1.png" style="width:80%"></center>

<center>图例：rhoaiserving-consuming 项目中的模型服务与推理 API</center>

<center><img src="images/model-serving-cr-2.png" style="width:80%"></center>

<center>图例：rhoaiserving-consuming 项目中的推理 API 由 route 发布</center>

#### **3.2.4 利用推理 API 验证模型**

**方法1**：使用 `curl` 命令以及模型推理 API 与访问用 Token 完成推理验证

**方法2**：使用 Python 的 requests 库完成推理认证

#### **3.2.5 KServe 与 ModelMesh 在多模型服务中的协作时序**

<center><img src="images/kserve-knative-rhoai-multimodel-cooperate.png" style="width:100%"></center>

<center>图例：KServe + ModelMesh 在 OpenShift AI 多模型服务中的协作时序</center>

用户可使用多模型服务提供的模型推理 API 访问模型，访问的完整流量链：

✔️ **user HTTP ➡ rest-proxy 容器 ➡ modelmesh 容器 (HTTP → gRPC 协议转换) ➡ model runtime 容器**

## 🌐 **4. 单模型服务与多模型服务中模型推理 API 的生成**

<center><img src="images/rhoai-single-multi-infer-url-generate.png" style="width:100%"></center>

## 🔎 **5. 参考链接**

- 🎇 [KServe](https://kserve.github.io/website/)
- [KServe | GitHub](https://github.com/kserve/kserve)
- [ModelMesh | GitHub](https://github.com/kserve/modelmesh)
- [Knative](https://knative.dev/)
- [Knative | GitHub](https://github.com/knative)
- [PlantUML 在线时序图绘制](https://www.plantuml.com/plantuml/uml/SyfFKj2rKt3CoKnELR1Io4ZDoSa700001)